# 08 — Phase 2: Query Node Feature Ablation

Staged ablation study, Phase 2 of 4. Ablates query node geometry features —
surface normal (3-vector) and mean curvature (scalar, log1p-scaled), injected
via `QueryEncoder` before AQ message passing — against a zeroed-features
baseline. Uses each architecture's **winning aggregation from Phase 1**
(notebook 07), locked in rather than re-swept.

**Status: not yet run.** Phase 1 must conclude first — this notebook is a
ready-to-fill scaffold, not live results.

## Selection methodology

Same as Phase 1: winners chosen from validation Pearson r (`metrics.csv`,
epoch of min val loss), RMSE as tie-break, watch train/val gap. Test metrics
are reference-only.

## Prerequisites

- [ ] Phase 1 (notebook 07) concluded, winning `agg` identified per architecture
- [ ] `sweeps/phase2_query_features_ablation.yaml` written, locking in each
      architecture's Phase 1 winner and toggling `query_curvature`/`query_normal`
      on vs off (4 runs: `{distance, attention} x {features_on, features_off}`)
- [ ] Sweep run: `python pipelines/run_sweep.py sweeps/phase2_query_features_ablation.yaml --all`
      (this **does** require a graph rebuild — `query_curvature`/`query_normal`
      change what's stored in the cached graph, unlike Phase 1's aggregation change)
- [ ] `scripts/analyze_model.py --curves --distributions --save-plots` run per checkpoint

## Decision

*To be filled in once Phase 2 completes.*


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve()))


def show_png_grid(runs, filename, title, ncols=2):
    """Display saved PNG plots from each run's plot_dir in a grid."""
    available = [r for r in runs if (r["plot_dir"] / filename).exists()]
    if not available:
        print(f"No '{filename}' plots found. Run analyze_model.py --save-plots first.")
        return
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
    axes = axes.flatten() if nrows * ncols > 1 else [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    for ax, run in zip(axes, available):
        ax.imshow(mpimg.imread(run["plot_dir"] / filename))
        ax.set_title(run["label"], fontsize=11)
        ax.axis("off")
    for ax in axes[len(available):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_metric_bars(df, title_prefix, metrics):
    """1-row x 2-col grouped bar chart: one chart per model type."""
    if df.empty:
        print("No data to plot.")
        return
    model_types = ["Attention", "Distance"]
    fig, axes = plt.subplots(1, len(model_types), figsize=(10, 5), sharey=False)
    fig.suptitle(title_prefix, fontsize=13, fontweight="bold")
    n_metrics = len(metrics)
    total_width = 0.6
    bar_w = total_width / n_metrics
    for ax, model_type in zip(axes, model_types):
        sub = df[df["Model"] == model_type].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        x = range(len(sub))
        for i, (metric, color) in enumerate(metrics):
            if metric not in sub.columns or sub[metric].isna().all():
                continue
            offsets = [xi - total_width / 2 + bar_w * i + bar_w / 2 for xi in x]
            bars = ax.bar(offsets, sub[metric], width=bar_w, color=color, label=metric, zorder=3)
            ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=90)
        ax.set_title(model_type, fontsize=12)
        ax.set_xlabel("Query features")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["Features"].tolist(), fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3, zorder=0)
    plt.tight_layout()
    plt.show()

## 1. Configuration

`WINNING_AGG` records each architecture's Phase 1 winner — fill in from
notebook 07's Section 7 decision before running the Phase 2 sweep. The
checkpoint dir naming (`<model>_<suffix>`) follows the same convention as
Phase 1.

In [ ]:
THESIS_ROOT = Path("/home/student/thesis")
CKPT_ROOT   = THESIS_ROOT / "checkpoints"
EVAL_ROOT   = THESIS_ROOT / "model_eval"

# Fill in from notebook 07's Phase 1 decision.
WINNING_AGG = {
    "attention": None,   # e.g. "multi"
    "distance":  None,   # e.g. "mean"
}

RUNS = [
    dict(label=f"{model.capitalize()} — features {onoff}", model_type=model, features=onoff,
         plot_dir=EVAL_ROOT/f"{model}_features_{onoff}", ckpt_dir=CKPT_ROOT/f"{model}_features_{onoff}")
    for model in ["attention", "distance"]
    for onoff in ["off", "on"]
]

print(f"{'Run':<28}  {'Plots':>6}  {'Metrics':>8}")
print("-" * 48)
for r in RUNS:
    has_plots   = r["plot_dir"].exists()
    has_metrics = (r["ckpt_dir"] / "metrics.csv").exists()
    print(f"{r['label']:<28}  {'yes' if has_plots else 'no':>6}  {'yes' if has_metrics else 'no':>8}")

## 2. Training Curves

Plots saved by `scripts/analyze_model.py --curves` for each run.

In [ ]:
show_png_grid(RUNS, "training_curves.png", "Training Curves — Query Feature Ablation")

## 3. Error Distributions

Plots saved by `scripts/analyze_model.py --distributions` for each run.

In [ ]:
show_png_grid(RUNS, "error_distributions.png", "Error Distributions — Query Feature Ablation", ncols=1)

## 4. Validation Metrics Comparison (selection basis)

Same methodology as Phase 1 — best-val-loss epoch from `metrics.csv`, not
`test_metrics.json`.

In [ ]:
rows = []
for run in RUNS:
    csv_path = run["ckpt_dir"] / "metrics.csv"
    if not csv_path.exists():
        continue
    hist = pd.read_csv(csv_path)
    if hist.empty:
        continue
    best = hist.loc[hist["val_loss"].idxmin()]
    rows.append({
        "Run":           run["label"],
        "Model":         run["model_type"].capitalize(),
        "Features":      run["features"],
        "Pearson r":     best["val_pearson_r"],
        "RMSE":          best["val_rmse"],
        "Val loss":      best["val_loss"],
        "Train loss":    best["train_loss"],
        "Train/val gap": best["train_loss"] - best["val_loss"],
        "Best epoch":    int(best["epoch"]),
    })

val_df = pd.DataFrame(rows).sort_values(["Model", "Pearson r"], ascending=[True, False])
pd.set_option("display.float_format", "{:.4f}".format)
display(val_df)

In [ ]:
plot_metric_bars(val_df, "Validation metrics — Query feature ablation (selection basis)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange")])

## 5. Test Metrics (reference only — not used for phase selection)

In [ ]:
rows_test = []
for run in RUNS:
    metrics_path = run["ckpt_dir"] / "test_metrics.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        data = json.load(f)
    g = data.get("global", {})
    rows_test.append({
        "Run":       run["label"],
        "Model":     run["model_type"].capitalize(),
        "Features":  run["features"],
        "Pearson r": g.get("pearson_r"),
        "RMSE":      g.get("rmse"),
        "MAE":       g.get("mae"),
        "N proteins": g.get("n_proteins"),
    })

test_df = pd.DataFrame(rows_test).sort_values(["Model", "Pearson r"], ascending=[True, False])
display(test_df)

In [ ]:
plot_metric_bars(test_df, "Test metrics — Query feature ablation (reference only)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange"), ("MAE", "seagreen")])

## 6. Error Distribution by ESP Value

Does adding query geometry features change whether the model struggles at
extreme positive/negative ESP values? (SUMMER_PLAN.md "Error Distribution by
ESP Value".) Plots saved by `scripts/analyze_model.py --error-by-esp` for
each run.

Left panel: residual vs ground-truth ESP, coloured by net charge, with a
binned median |residual| trend line. Right panel: residual histograms split
by |ESP| tertile. Compare `features on` vs `features off` — if the geometry
features narrow the "high |ESP|" residual distribution, that's a concrete
win for feature overload not showing up in aggregate Pearson r/RMSE alone.

In [ ]:
show_png_grid(RUNS, "error_by_esp.png", "Error Distribution by ESP Value — Query Feature Ablation", ncols=1)

## 7. Decision

*Fill in once Phase 2 completes. Template:*

| Model | Winning Phase 1 agg | Features on or off? | Val Pearson r | Val RMSE | Train/val gap |
|---|---|---|---|---|---|
| Attention | ? | ? | ? | ? | ? |
| Distance | ? | ? | ? | ? | ? |

**Carried forward to notebook 09 (equivariance reliance):** the winning
aggregation + feature config for each architecture, evaluated (no retraining)
under random SO(3) rotations and translations of the test set — see
`09_equivariance_reliance.ipynb`. The `features on` vs `features off`
checkpoints here are exactly the models that notebook tests, since
`query_normal` is the one input that isn't rotation-invariant by
construction (see that notebook's intro for why).

**Carried forward to Phase 3 (notebook 10):** *the winning aggregation +
feature config for each architecture, locked in for the protein-size-weighting
ablation sweep.*